# Beschreibung: 

# Importe:

In [2]:
import sys
import os

import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))
from rst_functions import discretize, indiscernibility, dependency, quick_reduct, induce_rules

os.getcwd()

'/home/samel/01. Projekte/01. Master/COMPARE_RST/Manuelle_Ausfuehrungen'

# Daten laden:
Heart Failure Prediction Dataset: https://www.kaggle.com/datasets/utkarsh1093/crime-data-from-2020-to-nov2025?select=Crime_Data_from_2020_to_Present.parquet

In [3]:
crimes = pd.read_parquet("../Daten/Crimes/LA_Crimes_2025.parquet")
print(crimes.shape)
print(crimes.columns)

(1004991, 28)
Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'LOCATION', 'LAT', 'LON', 'occ_year', 'occ_month', 'occ_date',
       'occ_day'],
      dtype='object')


In [3]:
# --- DEFINITELY categorical ---
cat_cols_definite = [
    'AREA', 'AREA NAME', 'Part 1-2',
    'Vict Sex', 'Vict Descent',
    'Status', 'Status Desc',
    'occ_year', 'occ_month', 'occ_date', 'occ_day'
]

# --- OPTIONAL categorical (still useful) ---
cat_cols_optional = [
    'Crm Cd', 'Crm Cd Desc',
    'Weapon Used Cd', 'Weapon Desc',
    'Premis Cd', 'Premis Desc'
]

# Convert to category dtype
for col in cat_cols_definite + cat_cols_optional:
    crimes[col] = crimes[col].astype('category')

# Ausführung:

### Vorbereitung: (Datenaufbereitung)

In [4]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "Weapon Used Cd" # Wenn nach den Umsänden gefragt, wo es Wahrscheinlich ist eine Waffe zu nutzen, oder 

In [5]:
cutoffs = {}

# Diskretisierung der numerischen Daten:
X = crimes.drop(columns=[decision_attr])

# ALLE Konditionsattribute diskretisieren (inkl. kategoriale)
crimes_disc = discretize(X, bins=4, cutoffs=cutoffs)

# Entscheidungsattribut wieder anhängen
crimes_disc[decision_attr] = crimes[decision_attr]
print(crimes_disc.columns)

Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1', 'LOCATION', 'LAT',
       'LON', 'occ_year', 'occ_month', 'occ_date', 'occ_day', 'DR_NO_disc',
       'AREA_disc', 'Rpt Dist No_disc', 'Part 1-2_disc', 'Crm Cd_disc',
       'Vict Age_disc', 'Premis Cd_disc', 'Crm Cd 1_disc', 'LAT_disc',
       'LON_disc', 'Weapon Used Cd'],
      dtype='object')


In [6]:
cond_attrs = []
for col in crimes_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte
        if col.endswith("_disc"):
            cond_attrs.append(col)
        # direkt kategorische Werte
        elif crimes_disc[col].dtype == "object":
            cond_attrs.append(col)
            
#cond_attrs.remove('DR_NO') # Das ist ein Identifier, daher muss es raus.
print(cond_attrs)

['TIME OCC', 'AREA NAME', 'Crm Cd Desc', 'Mocodes', 'Vict Sex', 'Vict Descent', 'Premis Desc', 'Weapon Desc', 'Status', 'Status Desc', 'LOCATION', 'occ_month', 'occ_day', 'DR_NO_disc', 'AREA_disc', 'Rpt Dist No_disc', 'Part 1-2_disc', 'Crm Cd_disc', 'Vict Age_disc', 'Premis Cd_disc', 'Crm Cd 1_disc', 'LAT_disc', 'LON_disc']


### Datenbetrachtung:

In [7]:
# Redukte
reduct, info = quick_reduct(crimes_disc, cond_attrs, decision_attr)

γ(C) mit allen Attributen: 1.000000
Einzel-γ-Werte:
  TIME OCC: γ = 0.000000
  AREA NAME: γ = 0.000000
  Crm Cd Desc: γ = 0.000574
  Mocodes: γ = 0.421706
  Vict Sex: γ = 0.000000
  Vict Descent: γ = 0.000002
  Premis Desc: γ = 0.000051
  Weapon Desc: γ = 1.000000
  Status: γ = 0.000001
  Status Desc: γ = 0.000000
  LOCATION: γ = 0.078564
  occ_month: γ = 0.000000
  occ_day: γ = 0.000000
  DR_NO_disc: γ = 0.000000
  AREA_disc: γ = 0.000000
  Rpt Dist No_disc: γ = 0.000000
  Part 1-2_disc: γ = 0.000000
  Crm Cd_disc: γ = 0.000000
  Vict Age_disc: γ = 0.000000
  Premis Cd_disc: γ = 0.000000
  Crm Cd 1_disc: γ = 0.000000
  LAT_disc: γ = 0.000000
  LON_disc: γ = 0.000000

Mindestens ein Attribut hat γ({a}) > 0 → benutze quick_reduct_monotone.


In [8]:
# Regeln
rules = induce_rules(crimes_disc, reduct, decision_attr)

# Resultate

In [9]:
print("Ergebnisse:")

print(f"\n{decision_attr}:\n{reduct}")
print(f"\nAnzahl Regeln: {len(rules)}\n")

for r in rules[:10]:
    print(r)

#print("Rules:", rules_pass_biased[2]) # Einzeln
#print("Rules:", rules_pass_biased) # Das wären alle

Ergebnisse:

Weapon Used Cd:
['Weapon Desc']

Anzahl Regeln: 80

{'premise': {'Weapon Desc': 'Unknown'}, 'decision': np.float64(0.0), 'support': 677744}
{'premise': {'Weapon Desc': 'KNIFE WITH BLADE 6INCHES OR LESS'}, 'decision': np.float64(200.0), 'support': 6838}
{'premise': {'Weapon Desc': 'UNKNOWN WEAPON/OTHER WEAPON'}, 'decision': np.float64(500.0), 'support': 36390}
{'premise': {'Weapon Desc': 'STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)'}, 'decision': np.float64(400.0), 'support': 174761}
{'premise': {'Weapon Desc': 'AIR PISTOL/REVOLVER/RIFLE/BB GUN'}, 'decision': np.float64(114.0), 'support': 1864}
{'premise': {'Weapon Desc': 'HAND GUN'}, 'decision': np.float64(102.0), 'support': 20182}
{'premise': {'Weapon Desc': 'UNKNOWN FIREARM'}, 'decision': np.float64(106.0), 'support': 6582}
{'premise': {'Weapon Desc': 'VERBAL THREAT'}, 'decision': np.float64(511.0), 'support': 23846}
{'premise': {'Weapon Desc': 'SEMI-AUTOMATIC PISTOL'}, 'decision': np.float64(109.0), 'support': 7267}
